In [ ]:
%pip install pandas


In [ ]:
dbutils.library.restartPython()


In [ ]:
import pandas as pd
df = pd.read_csv('/Volumes/insight/default/titanic/Titanic.csv')
df.head()


In [ ]:
print("DESCRIBE")
print("=" * 60)
print(df.describe())

print("\nMISSING VALUES")
print("=" * 60)
print(df.isnull().sum())

print("\nDUPLICATES")
print("=" * 60)
print(df.duplicated().sum())


In [ ]:
def get_basic_summary(df):
    """
    Returns a dictionary containing all important
    summary statistics about the DataFrame.
    Called by the Schema Agent in the pipeline.
    """
    summary = {
        "rows"                : df.shape[0],
        "columns"             : df.shape[1],
        "column_names"        : list(df.columns),
        "dtypes"              : df.dtypes.astype(str).to_dict(),
        "missing_values"      : df.isnull().sum().to_dict(),
        "missing_percent"     : (df.isnull().sum() / len(df) * 100).round(2).to_dict(),
        "duplicates"          : int(df.duplicated().sum()),
        "numeric_columns"     : list(df.select_dtypes(include="number").columns),
        "categorical_columns" : list(df.select_dtypes(include="object").columns),
        "total_missing_cells" : int(df.isnull().sum().sum()),
        "memory_usage_kb"     : round(df.memory_usage(deep=True).sum() / 1024, 2),
    }
    return summary

print("✅ get_basic_summary() function defined")


In [ ]:
def get_statistical_summary(df):
    """
    Returns detailed statistics for all numeric columns.
    Includes mean, median, std, skewness, kurtosis.
    """
    numeric_df = df.select_dtypes(include="number")

    stats = {}
    for col in numeric_df.columns:
        stats[col] = {
            "mean"    : round(numeric_df[col].mean(), 2),
            "median"  : round(numeric_df[col].median(), 2),
            "std"     : round(numeric_df[col].std(), 2),
            "min"     : round(numeric_df[col].min(), 2),
            "max"     : round(numeric_df[col].max(), 2),
            "skewness": round(numeric_df[col].skew(), 2),
            "kurtosis": round(numeric_df[col].kurt(), 2),
        }

    return stats

print("✅ get_statistical_summary() function defined")


In [ ]:
def get_correlation_matrix(df):
    """
    Returns correlation matrix for all numeric columns.
    Values close to 1  = strong positive relationship
    Values close to -1 = strong negative relationship
    Values close to 0  = no relationship
    """
    numeric_df = df.select_dtypes(include="number")

    if numeric_df.shape[1] < 2:
        print("⚠️ Need at least 2 numeric columns for correlation")
        return None

    corr_matrix = numeric_df.corr().round(2)
    return corr_matrix

print("✅ get_correlation_matrix() function defined")


In [ ]:
def get_value_counts(df, top_n=10):
    """
    Returns value counts for all categorical columns.
    Shows the top N most common values in each column.
    Used by the EDA Agent to understand categories.
    """
    cat_cols = df.select_dtypes(include="object").columns
    value_counts = {}

    for col in cat_cols:
        counts = df[col].value_counts().head(top_n)
        value_counts[col] = counts.to_dict()

    return value_counts

print("✅ get_value_counts() function defined")


In [ ]:
# ── 1. Basic Summary ──────────────────────────────────────────
print("=" * 50)
print("BASIC SUMMARY")
print("=" * 50)
summary = get_basic_summary(df)
for key, value in summary.items():
    print(f"  {key:25} → {value}")


In [ ]:
# ── 2. Statistical Summary ────────────────────────────────────
print("=" * 50)
print("STATISTICAL SUMMARY")
print("=" * 50)
stats = get_statistical_summary(df)
for col, values in stats.items():
    print(f"\n  📊 {col}")
    for stat_name, stat_val in values.items():
        print(f"      {stat_name:12} : {stat_val}")


In [ ]:
# ── 3. Correlation Matrix ─────────────────────────────────────
print("=" * 50)
print("CORRELATION MATRIX")
print("=" * 50)
corr = get_correlation_matrix(df)
print(corr)


In [ ]:
# ── 4. Value Counts ───────────────────────────────────────────

categorical_like_cols = [col for col in df.columns if df[col].nunique() <= 10]

for col in categorical_like_cols:
    print(f"\n📋 Column: {col}")
    counts = df[col].value_counts().head(10).reset_index()
    counts.columns = [col, "Count"]
    display(counts)


In [ ]:
# Run all 4 functions together
summary = get_basic_summary(df)
stats   = get_statistical_summary(df)
corr    = get_correlation_matrix(df)
vc      = get_value_counts(df)

print("BASIC SUMMARY")
print("=" * 40)
for key, value in summary.items():
    print(f"  {key:25} : {value}")

print("\n\nSTATISTICAL SUMMARY")
print("=" * 40)
for col, values in stats.items():
    print(f"\n  {col}")
    for stat_name, stat_val in values.items():
        print(f"    {stat_name:12} : {stat_val}")

print("\n\nCORRELATION MATRIX")
print("=" * 40)
print(corr.to_string())

print("\n\nVALUE COUNTS")
categorical_like_cols = [col for col in df.columns if df[col].nunique() <= 10]

for col in categorical_like_cols:
    print(f"\n📋 Column: {col}")
    counts = df[col].value_counts().head(10).reset_index()
    counts.columns = [col, "Count"]
    display(counts)


In [ ]:
def get_basic_summary(df):
    summary = {
        "rows"                : df.shape[0],
        "columns"             : df.shape[1],
        "column_names"        : list(df.columns),
        "dtypes"              : df.dtypes.astype(str).to_dict(),
        "missing_values"      : df.isnull().sum().to_dict(),
        "missing_percent"     : (df.isnull().sum() / len(df) * 100).round(2).to_dict(),
        "duplicates"          : int(df.duplicated().sum()),
        "numeric_columns"     : list(df.select_dtypes(include="number").columns),
        "categorical_columns" : list(df.select_dtypes(include="object").columns),
        "total_missing_cells" : int(df.isnull().sum().sum()),
        "memory_usage_kb"     : round(df.memory_usage(deep=True).sum() / 1024, 2),
    }
    return summary

print("✅ get_basic_summary defined")


In [ ]:
def get_statistical_summary(df):
    """
    Returns mean, median, std, skewness, kurtosis
    for every numeric column in the DataFrame.
    """
    numeric_df = df.select_dtypes(include="number")

    result = {}

    for col in numeric_df.columns:
        result[col] = {
            "mean"    : round(numeric_df[col].mean(),   2),
            "median"  : round(numeric_df[col].median(), 2),
            "std"     : round(numeric_df[col].std(),    2),
            "min"     : round(numeric_df[col].min(),    2),
            "max"     : round(numeric_df[col].max(),    2),
            "skewness": round(numeric_df[col].skew(),   2),
            "kurtosis": round(numeric_df[col].kurt(),   2),
        }

    return result

print("✅ get_statistical_summary defined")


In [ ]:
def get_correlation_matrix(df):
    """
    Returns a correlation matrix for all numeric columns.
    Only works when there are 2 or more numeric columns.
    """
    numeric_df = df.select_dtypes(include="number")

    if numeric_df.shape[1] < 2:
        print("⚠️ Need at least 2 numeric columns")
        return None

    corr = numeric_df.corr().round(2)
    return corr

print("✅ get_correlation_matrix defined")


In [ ]:
def get_value_counts(df, top_n=10):
    """
    Returns value counts for every categorical column.
    Stores results as a dict — no display() used
    so it works in every Databricks version.
    """
    cat_cols = df.select_dtypes(include="object").columns
    result   = {}

    for col in cat_cols:
        counts       = df[col].value_counts().head(top_n)
        result[col]  = counts.to_dict()

    return result

print("✅ get_value_counts defined")


In [ ]:
# ── Call all 4 functions ──────────────────────────────────────
summary = get_basic_summary(df)
stats   = get_statistical_summary(df)
corr    = get_correlation_matrix(df)
vc      = get_value_counts(df)

# ════════════════════════════════════════════════════════════
print("1.  BASIC SUMMARY")
print("=" * 55)
print(f"  Rows                 : {summary['rows']}")
print(f"  Columns              : {summary['columns']}")
print(f"  Duplicates           : {summary['duplicates']}")
print(f"  Total missing cells  : {summary['total_missing_cells']}")
print(f"  Memory               : {summary['memory_usage_kb']} KB")
print(f"  Numeric columns      : {summary['numeric_columns']}")
print(f"  Categorical columns  : {summary['categorical_columns']}")
print(f"\n  Missing per column:")
for col, count in summary["missing_values"].items():
    pct = summary["missing_percent"][col]
    if count > 0:
        bar = "█" * int(pct / 5)
        print(f"    {col:15} : {count:4} missing  ({pct}%)  {bar}")
    else:
        print(f"    {col:15} : no missing values")

# ════════════════════════════════════════════════════════════
print("\n\n2.  STATISTICAL SUMMARY")
print("=" * 55)
for col, values in stats.items():
    print(f"\n  {col}")
    print(f"    mean      : {values['mean']}")
    print(f"    median    : {values['median']}")
    print(f"    std       : {values['std']}")
    print(f"    min       : {values['min']}")
    print(f"    max       : {values['max']}")
    print(f"    skewness  : {values['skewness']}")
    print(f"    kurtosis  : {values['kurtosis']}")

# ════════════════════════════════════════════════════════════
print("\n\n3.  CORRELATION MATRIX")
print("=" * 55)
print(corr.to_string())
print()
print("  Key relationships:")
print(f"  2urvived vs Pclass : {corr.loc['2urvived','Pclass']}  → negative means higher class = more survival")
print(f"  2urvived vs Fare   : {corr.loc['2urvived','Fare']}   → positive means higher fare = more survival")
print(f"  2urvived vs Age    : {corr.loc['2urvived','Age']}  → weak negative means younger survived slightly more")

# ════════════════════════════════════════════════════════════
# 4. VALUE COUNTS
print("\n\n4. VALUE COUNTS")
print("=" * 55)

# Select columns with 10 or fewer unique values, excluding constant zero columns
categorical_cols = [
    col for col in df.columns 
    if df[col].nunique() <= 10 and not (df[col] == 0).all()
]

total_rows = summary["rows"]

for col in categorical_cols:
    print(f"\n📋 {col}")
    counts = df[col].value_counts().items()
    
    for val, count in counts:
        pct = round((count / total_rows) * 100, 1)
        bar = "█" * int(pct / 5)
        print(f"  {str(val):<20} : {count:<4} ({pct:>5.1f}%) {bar}")
